<a href="https://colab.research.google.com/github/viltegonzalo/AplicacionBaseDatos-II/blob/main/TP7/TP7_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1: Instalación de dependencias e Importación de librerías

In [12]:
# Instalar paquetes necesarios (gdown para descargar de Drive, mlxtend para Apriori)
!pip install -q gdown mlxtend
import gdown
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore') # Para evitar warnings de formato

## Paso 0 - Carga del Dataset desde Google Drive

In [13]:
from google.colab import drive
import pandas as pd

# 1. Esto te pedirá permisos para acceder a tu Google Drive
print("Montando Google Drive...")
drive.mount('/content/drive')

# 2. Ruta al archivo.
# IMPORTANTE: Cambia esta ruta si guardaste el archivo dentro de una carpeta específica.
# Por defecto asume que está suelto en "Mi unidad" (My Drive)
ruta_archivo = '/content/drive/MyDrive/FACULTAD/Carreras/Materias en Comun/Aplicaciones de Base de Datos II/Trabajo Grupal ADB-II/TP-7/DataSet/retail_dataset.csv'

# 3. Leer el archivo
df = pd.read_csv(ruta_archivo)

# 4. Mostrar primeras filas y resumen
print("\n--- Primeras 5 filas del Dataset Original ---")
display(df.head())

print("\n--- Resumen Informativo del Dataset ---")
df.info()



Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Primeras 5 filas del Dataset Original ---


,0,1,2,3,4,5,6
0,Bread,Wine,Eggs,Meat,Cheese,Pencil,Diaper
1,Bread,Cheese,Meat,Diaper,Wine,Milk,Pencil
2,Cheese,Meat,Eggs,Milk,Wine,NaN,NaN
3,Cheese,Meat,Eggs,Milk,Wine,NaN,NaN
4,Meat,Pencil,Wine,NaN,NaN,NaN,NaN



--- Resumen Informativo del Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       315 non-null    object
 1   1       285 non-null    object
 2   2       245 non-null    object
 3   3       187 non-null    object
 4   4       133 non-null    object
 5   5       71 non-null     object
 6   6       41 non-null     object
dtypes: object(7)
memory usage: 17.4+ KB


## Paso 2.a - Traducción de productos al Castellano

In [14]:
# 1. Obtener todos los valores únicos del dataset (ignorando valores nulos/NaN)
valores_unicos = pd.unique(df.values.ravel('K'))
productos_ingles = [prod for prod in valores_unicos if pd.notna(prod)]
print(f"Productos detectados en el dataset: {productos_ingles}\n")

# 2. Diccionario de traducción (Inglés -> Español)
traduccion = {
    'Bread': 'Pan',
    'Wine': 'Vino',
    'Cheese': 'Queso',
    'Meat': 'Carne',
    'Eggs': 'Huevos',
    'Diaper': 'Pañales',
    'Milk': 'Leche',
    'Pencil': 'Lápiz',
    'Bagel': 'Panecillo'
}

# 3. Aplicar traducción a todo el dataframe
# El método replace modificará los valores exactos, los NaN quedarán intactos
df_es = df.replace(traduccion)

print("--- Primeras 5 filas del Dataset Traducido ---")
display(df_es.head())


Productos detectados en el dataset: ['Bread', 'Cheese', 'Meat', 'Eggs', 'Wine', 'Bagel', 'Pencil', 'Diaper', 'Milk']

--- Primeras 5 filas del Dataset Traducido ---


,0,1,2,3,4,5,6
0,Pan,Vino,Huevos,Carne,Queso,Lápiz,Pañales
1,Pan,Queso,Carne,Pañales,Vino,Leche,Lápiz
2,Queso,Carne,Huevos,Leche,Vino,NaN,NaN
3,Queso,Carne,Huevos,Leche,Vino,NaN,NaN
4,Carne,Lápiz,Vino,NaN,NaN,NaN,NaN


## Paso 2.b - Transformación a Formato Tidy Data (Matriz Binaria)

In [15]:
# 1. Convertir el DataFrame en una Lista de Listas (Transacciones)
# Necesitamos iterar por filas y descartar los valores nulos (NaN)
transacciones = []
for index, fila in df_es.iterrows():
    # Extraer valores, quitar nulos y convertir a lista de strings
    transaccion = [str(item) for item in fila.dropna().values]
    if transaccion: # Si la transacción no está vacía
        transacciones.append(transaccion)
# 2. Instanciar TransactionEncoder y transformar los datos
te = TransactionEncoder()
# Fit extrae los nombres únicos y transform crea la matriz binaria (True/False)
matriz_binaria = te.fit(transacciones).transform(transacciones)
# 3. Crear el DataFrame en formato Tidy Data
df_tidy = pd.DataFrame(matriz_binaria, columns=te.columns_)
print("--- Matriz Transaccional Binaria (One-Hot Encoding) ---")
display(df_tidy.head())

--- Matriz Transaccional Binaria (One-Hot Encoding) ---


,Carne,Huevos,Leche,Lápiz,Pan,Panecillo,Pañales,Queso,Vino
0,True,True,False,True,True,False,True,True,True
1,True,False,True,True,True,False,True,True,True
2,True,True,True,False,False,False,False,True,True
3,True,True,True,False,False,False,False,True,True
4,True,False,False,True,False,False,False,False,True


## Paso 2.c - Aplicación del Algoritmo A priori

In [16]:
# 1. Extraer Itemsets Frecuentes usando Apriori
# Fijamos el soporte mínimo en 0.15 (el producto o combinación debe aparecer en al menos el 15% de las compras)
itemsets_frecuentes = apriori(df_tidy, min_support=0.15, use_colnames=True)
# 2. Generar Reglas de Asociación
# Extraemos reglas que tengan una Confianza mínima del 50% (0.5)
reglas = association_rules(itemsets_frecuentes, metric="confidence", min_threshold=0.5)
# 3. Ordenar las reglas por Lift (fuerza de la regla) de mayor a menor
reglas_ordenadas = reglas.sort_values(by="lift", ascending=False)
print(f"Se encontraron un total de {len(reglas_ordenadas)} reglas con Confianza >= 50% y Soporte >= 15%\n")
print("--- Top Reglas de Asociación (Ordenadas por Lift) ---")
display(reglas_ordenadas.head(10)) # Mostramos las 10 mejores

Se encontraron un total de 62 reglas con Confianza >= 50% y Soporte >= 15%

--- Top Reglas de Asociación (Ordenadas por Lift) ---


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
61,"(Leche, Carne)","(Huevos, Queso)",0.244444,0.298413,0.152381,0.623377,2.088975,1.0,0.079436,1.862835,0.689951,0.390244,0.463184,0.567007
57,"(Huevos, Queso)","(Leche, Carne)",0.298413,0.244444,0.152381,0.510638,2.088975,1.0,0.079436,1.543961,0.743024,0.390244,0.352315,0.567007
58,"(Huevos, Leche)","(Queso, Carne)",0.244444,0.323810,0.152381,0.623377,1.925134,1.0,0.073228,1.795402,0.636029,0.366412,0.443022,0.546982
59,"(Huevos, Carne)","(Queso, Leche)",0.266667,0.304762,0.152381,0.571429,1.875000,1.0,0.071111,1.622222,0.636364,0.363636,0.383562,0.535714
60,"(Queso, Leche)","(Huevos, Carne)",0.304762,0.266667,0.152381,0.500000,1.875000,1.0,0.071111,1.466667,0.671233,0.363636,0.318182,0.535714
56,"(Queso, Leche, Carne)",(Huevos),0.203175,0.438095,0.152381,0.750000,1.711957,1.0,0.063371,2.247619,0.521912,0.311688,0.555085,0.548913
55,"(Huevos, Leche, Carne)",(Queso),0.177778,0.501587,0.152381,0.857143,1.708861,1.0,0.063210,3.488889,0.504505,0.289157,0.713376,0.580470
31,"(Leche, Carne)",(Huevos),0.244444,0.438095,0.177778,0.727273,1.660079,1.0,0.070688,2.060317,0.526261,0.352201,0.514638,0.566535
37,"(Leche, Carne)",(Queso),0.244444,0.501587,0.203175,0.831169,1.657077,1.0,0.080564,2.952137,0.524816,0.374269,0.661262,0.618116
53,"(Huevos, Queso, Leche)",(Carne),0.196825,0.476190,0.152381,0.774194,1.625806,1.0,0.058655,2.319728,0.479249,0.292683,0.568915,0.547097


## Paso 2.d - Selección y Explicación de Reglas de Interés
En esta celda generamos dinámicamente un reporte de negocio para que el equipo entienda los hallazgos sin saber estadística matemática.

In [17]:
# Seleccionamos las 3 mejores reglas basadas en el Lift y la Confianza
top_3_reglas = reglas_ordenadas.head(3).reset_index(drop=True)

print("="*80)
print("📊 ANÁLISIS DE NEGOCIO: LAS REGLAS DE COMPRA MÁS RELEVANTES 📊")
print("="*80)

for i, fila in top_3_reglas.iterrows():
    # Convertimos los frozensets a strings legibles
    antecedente = ", ".join(list(fila['antecedents']))
    consecuente = ", ".join(list(fila['consequents']))

    # Métricas
    soporte_pct = fila['support'] * 100
    confianza_pct = fila['confidence'] * 100
    lift_val = fila['lift']

    print(f"\n🌟 REGLA #{i+1}: Si el cliente compra [{antecedente}], entonces también compra [{consecuente}]")
    print(f"   ▶ Soporte ({soporte_pct:.2f}%):")
    print(f"      Significado: Esta combinación de productos ({antecedente} + {consecuente}) aparece ")
    print(f"      en el {soporte_pct:.2f}% de TODAS las transacciones registradas de la tienda.")
    print(f"      Es un patrón recurrente en el volumen total de ventas.")

    print(f"   ▶ Confianza ({confianza_pct:.2f}%):")
    print(f"      Significado: De todas las veces que un cliente llevó la canasta con [{antecedente}], ")
    print(f"      en un {confianza_pct:.2f}% de los casos terminó agregando también [{consecuente}].")
    print(f"      ¡Existe una probabilidad muy alta de que ocurra esta venta cruzada!")

    print(f"   ▶ Lift ({lift_val:.2f}):")
    print(f"      Significado: La compra de [{antecedente}] aumenta en un factor de {lift_val:.2f} veces ")
    print(f"      la probabilidad de que se compre [{consecuente}], en comparación a si los artículos ")
    print(f"      se compraran de forma independiente. Al ser un Lift > 1, la asociación es Fuerte y Positiva.")
    print("-" * 80)


📊 ANÁLISIS DE NEGOCIO: LAS REGLAS DE COMPRA MÁS RELEVANTES 📊

🌟 REGLA #1: Si el cliente compra [Leche, Carne], entonces también compra [Huevos, Queso]
   ▶ Soporte (15.24%):
      Significado: Esta combinación de productos (Leche, Carne + Huevos, Queso) aparece 
      en el 15.24% de TODAS las transacciones registradas de la tienda.
      Es un patrón recurrente en el volumen total de ventas.
   ▶ Confianza (62.34%):
      Significado: De todas las veces que un cliente llevó la canasta con [Leche, Carne], 
      en un 62.34% de los casos terminó agregando también [Huevos, Queso].
      ¡Existe una probabilidad muy alta de que ocurra esta venta cruzada!
   ▶ Lift (2.09):
      Significado: La compra de [Leche, Carne] aumenta en un factor de 2.09 veces 
      la probabilidad de que se compre [Huevos, Queso], en comparación a si los artículos 
      se compraran de forma independiente. Al ser un Lift > 1, la asociación es Fuerte y Positiva.
------------------------------------------------